# Cổng 0 – chạy TabPFN v2 cho G0-2 (cần GPU)

TabPFN v2 là baseline chỉ-dùng-dữ-liệu-đích mạnh nhất mà cô yêu cầu. Phần LR, RF, XGBoost và các baseline chuyển giao đã chạy xong và nằm sẵn trong `gate0_kb1_v2.zip`; notebook này chỉ chạy thêm TabPFN v2 **trên đúng 30 lần chia đó** rồi tính lại kết luận GO/PIVOT.

**Trước khi chạy: bật GPU.** Menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**.

Sau đó bấm ▶ lần lượt từng ô (hoặc **Runtime → Run all**). Tổng thời gian khoảng 20–40 phút.

## Ô 1 – Tải file lên
Bấm ▶, rồi bấm **Choose Files**, giữ Ctrl chọn cả 3 file: `gate0_kb1_v2.zip` và 2 file CSV (miền A – IBM, miền B – Job Change). Tên file đặt thế nào cũng được.

In [ ]:
import os, glob, zipfile, shutil
os.chdir('/content')
shutil.rmtree('/content/up', ignore_errors=True); os.makedirs('/content/up')
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    open(f'/content/up/{name}', 'wb').write(data)
print('Đã nhận:', list(uploaded))

## Ô 2 – Giải nén và nhận dạng file
Phải thấy 3 dòng **OK**.

Nếu dòng đầu báo **THIẾU Mã gate0_kb1_v2 (bản mới…)**: file zip bạn chọn là bản cũ. Tải lại bản `gate0_kb1_v2.zip` mới (khoảng 2,3 MB) ở Ô 1.

In [ ]:
import os, glob, zipfile, shutil, csv
os.chdir('/content')
for i, z in enumerate(glob.glob('/content/up/*.zip')):
    with zipfile.ZipFile(z) as f:
        f.extractall(f'/content/up/x{i}')
# Nhận dạng theo NỘI DUNG (dòng tiêu đề), không phụ thuộc tên file
ibm = jc = None
for p in sorted(glob.glob('/content/up/**/*.csv', recursive=True)):
    with open(p, encoding='utf-8-sig', errors='ignore') as f:
        cols = next(csv.reader(f), [])
    if 'Attrition' in cols and 'TotalWorkingYears' in cols and not ibm:
        ibm = p
    elif 'enrollee_id' in cols and ('target' in cols or 'job_chnge' in cols) and not jc:
        jc = p
k = sorted(glob.glob('/content/up/**/g0_23.py', recursive=True))   # chỉ bản mã mới có g0_23.py
code_dir = os.path.dirname(k[0]) if k else None
ok = True
for label, p in [('Mã gate0_kb1_v2 (bản mới, có g0_23.py)', code_dir), ('Miền A – IBM HR (có cột Attrition)', ibm), ('Miền B – Job Change (có cột enrollee_id, target)', jc)]:
    print(('OK    ' if p else 'THIẾU ') + label + ('  ->  ' + os.path.basename(p) if p else ''))
    ok = ok and bool(p)
if ok:
    shutil.rmtree('/content/gate0', ignore_errors=True)
    shutil.copytree(code_dir, '/content/gate0')
    os.makedirs('/content/gate0/data', exist_ok=True)
    shutil.copy(ibm, '/content/gate0/data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
    shutil.copy(jc, '/content/gate0/data/aug_train.csv')
    print('\nSẵn sàng. Chạy tiếp Ô 3.')

## Ô 3 – Kiểm tra GPU và cài TabPFN v2
Dòng đầu phải in tên GPU (ví dụ *Tesla T4*). Nếu báo không có GPU, xem lại bước bật GPU ở đầu notebook.

In [ ]:
!nvidia-smi -L || echo "KHÔNG CÓ GPU – vào Runtime → Change runtime type → T4 GPU"
!pip -q install "tabpfn==2.1.3"
import torch; print('CUDA:', torch.cuda.is_available())

## Ô 4 – Chạy TabPFN v2 trên 30 lần chia
In tiến độ từng lần chia. Kết quả được lưu dần, nên nếu phiên bị ngắt thì chạy lại ô này (các lần đã xong vẫn còn trong file).

Nếu lần đầu báo lỗi tải trọng số từ Hugging Face (401/403), chạy ô *Ô 4b* bên dưới để đăng nhập Hugging Face rồi chạy lại Ô 4.

In [ ]:
%cd /content/gate0
!python g0_23.py --ibm data/WA_Fn-UseC_-HR-Employee-Attrition.csv --jc data/aug_train.csv --only TabPFN --device cuda 2>&1 | grep -v -i warning

### Ô 4b – (chỉ khi Ô 4 báo lỗi tải trọng số)
Tạo token miễn phí tại huggingface.co → Settings → Access Tokens, mở trang `huggingface.co/Prior-Labs/TabPFN-v2-clf` bấm đồng ý điều khoản nếu có, rồi dán token khi ô hỏi.

In [ ]:
from huggingface_hub import login
login()

## Ô 5 – Tính lại kết luận GO/PIVOT (đã có TabPFN v2)
Dòng **KẾT LUẬN** ở cuối là kết quả chính thức của Cổng 0.

In [ ]:
%cd /content/gate0
!python g0_decision.py results/g0_23

## Ô 6 – Tải kết quả về máy
Tải về `ket_qua_g0_23.zip`; gửi file này cho Claude để cập nhật báo cáo.

In [ ]:
import shutil, os
shutil.make_archive('/content/ket_qua_g0_23', 'zip', '/content/gate0/results', 'g0_23')
from google.colab import files
files.download('/content/ket_qua_g0_23.zip')